# Chronos Forecasting Pipeline

Runs Chronos quantile prediction across three time horizons
(**Tune 1 / Tune 2 / Inference**) and compares each forecast against the
**Actuals** plus the **benchmark models defined per horizon** in `metric_df`,
using **MAPE**.

### Inputs
- `data_df`: wide-format actuals, indexed by Python `date`, with a target
  column and any covariate columns. Also exposes `DATE` as a column.
- `metric_df`: long-format table with columns
  `FORECAST_TYPE, PORTFOLIO, SUB_PORTFOLIO, METRIC, DATE, METRIC_VALUE`.
  `"Actual"` is the actuals label; everything else counts as a benchmark.

### Smoothing convention
The **target** sent to Chronos is moving-average smoothed.
Engineered features (lags / SMA / EMA / MACD) are computed on the **raw**
target on purpose — features track real dynamics while the target Chronos
forecasts is denoised.


## 1. Imports

In [ ]:
import json
from datetime import date
from typing import Optional

import boto3
import numpy as np
import pandas as pd

## 2. Paths and configuration

Benchmark labels map each horizon to the `FORECAST_TYPE` names in
`metric_df` that should be compared against Chronos.

In [ ]:
# -- Paths and benchmark source --
DATA_DIR     = "./data"
S3_BUCKET    = "fmsp-sagemaker-rstudio-01"
S3_FOLDER    = "Forecasting/Toyota"
METRICS_FILE = "Forecasting_Toyota_combined_with_predata_v1.parquet"

# -- Benchmark labels for each horizon (must match FORECAST_TYPE in metric_df) --
BENCHMARK_LABELS_TUNE_1 = [
    "2023 0+12",
    "Original",
]
BENCHMARK_LABELS_TUNE_2 = [
    "2024 0+12",
    "Original",
]
BENCHMARK_LABELS_INFER = [
    "2025 0+12",
    "Original",
]

# -- Forecasting settings --
FORECASTING_METRIC = "Gross Credit Losses"   # target column / METRIC value
PORTFOLIO          = "Toyota"
SUB_PORTFOLIO      = "Lexus CB"
ENDPOINT_NAME      = "jumpstart-dft-pt-forecasting-chrono-20260608-054056"
MA_WINDOW          = 3
PREDICTION_LENGTH  = 12   # full test horizon

# -- Feature engineering windows --
LAG_PERIODS  = [1, 2, 3, 12]
SMA_WINDOWS  = [3, 5, 7]
EMA_SPANS    = [3, 5, 7]
MACD_CONFIGS = [(3, 8, 3), (5, 10, 3)]   # (short_span, long_span, signal_span)

ACTUAL_LABEL      = "Actual"
DEFAULT_QUANTILES = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)

## 3. Load data

`metric_df` — long-format table from S3 (all FORECAST_TYPEs, all METRICs).  
`data_df` — wide-format actuals only (one row per month, one column per metric),
used by the feature engineering and horizon slicing steps.


In [ ]:
metric_df = load_benchmark_data(
    bucket=S3_BUCKET,
    folder=S3_FOLDER,
    metrics_file=METRICS_FILE,
)

# Construct wide-format actuals from metric_df.
# Each METRIC becomes a column; index is Python date.
data_df = (
    metric_df[
        (metric_df["FORECAST_TYPE"] == ACTUAL_LABEL) &
        (metric_df["PORTFOLIO"]     == PORTFOLIO) &
        (metric_df["SUB_PORTFOLIO"] == SUB_PORTFOLIO)
    ]
    .pivot(index="DATE", columns="METRIC", values="METRIC_VALUE")
    .sort_index()
)
data_df.index = pd.to_datetime(data_df.index).date
data_df["DATE"] = data_df.index   # keep DATE as a column too (used by feature builders)

print(f"Actual data: {len(data_df)} rows  ({data_df.index[0]} -> {data_df.index[-1]})")
print(f"Benchmark rows: {len(metric_df)}")
data_df.head()

## 4. Horizon definitions

Each dict carries `train_start / train_end / test_start / test_end` (ISO strings)
**plus** `benchmark_names` — the FORECAST_TYPE values from `metric_df` to compare
against for that specific horizon.


In [ ]:
HORIZONS = [
    {
        "label":           "Tune 1",
        "train_start":     "2017-04-30",
        "train_end":       "2022-12-31",
        "test_start":      "2023-01-31",
        "test_end":        "2023-12-31",
        "benchmark_names": BENCHMARK_LABELS_TUNE_1,
    },
    {
        "label":           "Tune 2",
        "train_start":     "2017-04-30",
        "train_end":       "2023-12-31",
        "test_start":      "2024-01-31",
        "test_end":        "2024-12-31",
        "benchmark_names": BENCHMARK_LABELS_TUNE_2,
    },
    {
        "label":           "Inference",
        "train_start":     "2017-04-30",
        "train_end":       "2024-12-31",
        "test_start":      "2025-01-31",
        "test_end":        "2025-12-31",
        "benchmark_names": BENCHMARK_LABELS_INFER,
    },
]

# Quick sanity check
for h in HORIZONS:
    test_months = pd.date_range(h["test_start"], h["test_end"], freq="ME")
    print(f"{h['label']:10s}  train: {h['train_start']} -> {h['train_end']}  "
          f"test: {h['test_start']} -> {h['test_end']}  ({len(test_months)} months)  "
          f"benchmarks: {h['benchmark_names']}")

## 5. Horizon slicing helpers

`slice_horizon` splits `data_df` into `(train_df, test_df)` for one horizon dict.

In [ ]:
def build_test_index(horizon: dict) -> pd.Index:
    """Monthly DateIndex for the horizon's test window (last day of each month)."""
    return pd.date_range(horizon["test_start"], horizon["test_end"], freq="ME").date


def slice_horizon(data_df: pd.DataFrame, horizon: dict
                  ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split `data_df` into (train_df, test_df) for one horizon."""
    train_start = date.fromisoformat(horizon["train_start"])
    train_end   = date.fromisoformat(horizon["train_end"])

    train_df = data_df[(data_df.index >= train_start) & (data_df.index <= train_end)]
    test_df  = data_df.reindex(build_test_index(horizon))
    return train_df, test_df

## 6. Feature engineering

Each helper does one thing and mutates the DataFrame in place.
Features are computed on the **raw** target column — only the target series
sent to Chronos is smoothed.

In [ ]:
def smooth_target(series: pd.Series, ma_window: int) -> pd.Series:
    """Moving-average smoothing. `ma_window <= 0` means no smoothing."""
    if ma_window and ma_window > 0:
        return series.rolling(window=ma_window).mean().dropna()
    return series


def add_lag_features(df, target_col, lags):
    for lag in lags:
        df[f"target_lag_{lag}"] = df[target_col].shift(lag)


def add_moving_average_features(df, target_col, sma_windows, ema_spans):
    for w in sma_windows:
        df[f"sma_{w}"] = df[target_col].rolling(window=w).mean()
    for s in ema_spans:
        df[f"ema_{s}"] = df[target_col].ewm(span=s, adjust=False).mean()


def add_seasonality_features(df, date_col="DATE"):
    """Cyclic month encoding so Dec (12) and Jan (1) are adjacent in feature space."""
    month = df[date_col].dt.month
    df["month"]     = month
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)


def add_macd_features(df, target_col, configs):
    """MACD = short EMA - long EMA; signal = EMA(MACD)."""
    for short, long, signal in configs:
        ema_short = df[target_col].ewm(span=short).mean()
        ema_long  = df[target_col].ewm(span=long).mean()
        macd      = ema_short - ema_long
        df[f"macd_{short}_{long}"]        = macd
        df[f"macd_signal_{short}_{long}"] = macd.ewm(span=signal).mean()

### `data_prep` — orchestrates one training slice

Smooths the target, trims leading rows, then attaches all engineered features.

Returns:
- `train_df` — original train slice with all feature columns added
- `train_series` — smoothed target as a list (what Chronos forecasts)
- `target_index` — positional index aligned to `train_series`, used to pick
  matching covariate rows from `train_df`


In [ ]:
def data_prep(train_df: pd.DataFrame,
              target_col: str,
              ma_window: int = 3,
              keep_last_n: Optional[int] = None,
              ) -> tuple[pd.DataFrame, list[float], pd.Index]:
    train_df = train_df.reset_index(drop=True).copy()

    # Target: smooth, then trim so length == len(train_df) - ma_window.
    smoothed = smooth_target(train_df[target_col], ma_window)
    if keep_last_n is None:
        keep_last_n = len(train_df) - ma_window
    smoothed = smoothed.iloc[-keep_last_n:]

    target_index = smoothed.index
    train_series = smoothed.tolist()

    # Features (all on raw target)
    add_lag_features(train_df, target_col, LAG_PERIODS)
    add_moving_average_features(train_df, target_col, SMA_WINDOWS, EMA_SPANS)
    add_seasonality_features(train_df, date_col="DATE")
    add_macd_features(train_df, target_col, MACD_CONFIGS)

    return train_df, train_series, target_index

## 7. Chronos endpoint call

Sends the target + past covariates to SageMaker and returns a dict of
quantile arrays, e.g. `{"0.5": np.array([...]), "0.3": ..., ...}`.

In [ ]:
def chronos_predict_quantiles(
    runtime,
    endpoint_name: str,
    target_series: list[float],
    past_covariates: list[list[float]],
    prediction_length: int = PREDICTION_LENGTH,
    quantiles: tuple[float, ...] = DEFAULT_QUANTILES,
) -> dict[str, np.ndarray]:
    payload = {
        "inputs": [{
            "target": target_series,
            "past_covariates": past_covariates,
        }],
        "parameters": {
            "prediction_length": prediction_length,
            "quantile_levels": list(quantiles),
        },
    }
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    result = json.loads(response["Body"].read().decode())
    predictions = result["predictions"][0]
    return {str(q): np.array(predictions[str(q)]) for q in quantiles}

## 8. Benchmark utilities

`fetch_metric_series` pulls one value series out of the long-format `metric_df`
for a specific `(FORECAST_TYPE, METRIC, PORTFOLIO, SUB_PORTFOLIO)` combination,
aligned to the requested test dates. Missing dates become `NaN`.

In [ ]:
def fetch_metric_series(metric_df: pd.DataFrame, *,
                        forecast_type: str,
                        metric: str,
                        portfolio: str,
                        sub_portfolio: str,
                        dates) -> pd.Series:
    target_dates = {pd.Timestamp(d).date() for d in dates}

    mask = (
        (metric_df["FORECAST_TYPE"] == forecast_type)
        & (metric_df["METRIC"]        == metric)
        & (metric_df["PORTFOLIO"]     == portfolio)
        & (metric_df["SUB_PORTFOLIO"] == sub_portfolio)
    )
    sub = metric_df.loc[mask, ["DATE", "METRIC_VALUE"]].copy()
    sub["DATE"] = pd.to_datetime(sub["DATE"]).dt.date
    sub = sub[sub["DATE"].isin(target_dates)].set_index("DATE")["METRIC_VALUE"]
    return sub.reindex(dates)

## 9. MAPE

Drops entries where actual is zero (avoids division blow-up) or where either
side is `NaN` (handles missing benchmark rows gracefully).

In [ ]:
def mape(actual, predicted) -> float:
    """Mean Absolute Percentage Error (%)."""
    actual    = np.asarray(actual,    dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid = (~np.isnan(actual)) & (~np.isnan(predicted)) & (actual != 0)
    if not valid.any():
        return np.nan
    return float(np.mean(np.abs((actual[valid] - predicted[valid]) / actual[valid])) * 100)

## 10. Per-horizon runner

Six steps:
1. Slice train window from `data_df`.
2. Build smoothed target + engineered features.
3. Pack `past_covariates` from the chosen covariate columns.
4. Call Chronos.
5. Pull actuals + horizon-specific benchmarks from `metric_df`.
6. Compute MAPE for Chronos and each benchmark; return a `result_df`.


In [ ]:
def run_one_horizon(
    *,
    data_df: pd.DataFrame,
    metric_df: pd.DataFrame,
    horizon: dict,
    runtime,
    endpoint_name: str,
    target_col: str,
    portfolio: str,
    sub_portfolio: str,
    covariate_cols: Optional[list[str]] = None,
    ma_window: int = MA_WINDOW,
    prediction_length: int = PREDICTION_LENGTH,
    quantiles: tuple[float, ...] = DEFAULT_QUANTILES,
) -> dict:
    # benchmark names come from the horizon dict
    benchmark_names = horizon.get("benchmark_names", [])

    # 1. Slice + prepare
    train_df, _ = slice_horizon(data_df, horizon)
    train_df = train_df.copy()
    train_df["DATE"] = pd.to_datetime(train_df["DATE"])
    train_df, train_series, target_index = data_prep(train_df, target_col, ma_window)

    # 2. Past covariates (shape: n_channels x T)
    past_covariates: list[list[float]] = []
    if covariate_cols:
        past_covariates = (
            train_df.loc[target_index, covariate_cols]
                    .ffill().values.T.tolist()
        )

    # 3. Chronos forecast
    forecast = chronos_predict_quantiles(
        runtime=runtime,
        endpoint_name=endpoint_name,
        target_series=train_series,
        past_covariates=past_covariates,
        prediction_length=prediction_length,
        quantiles=quantiles,
    )
    chronos_median = forecast["0.5"]

    # 4. Actuals + benchmarks for the test window
    test_dates = build_test_index(horizon)

    actuals = fetch_metric_series(
        metric_df,
        forecast_type=ACTUAL_LABEL,
        metric=target_col,
        portfolio=portfolio,
        sub_portfolio=sub_portfolio,
        dates=test_dates,
    )
    benchmark_series = {
        name: fetch_metric_series(
            metric_df,
            forecast_type=name,
            metric=target_col,
            portfolio=portfolio,
            sub_portfolio=sub_portfolio,
            dates=test_dates,
        )
        for name in benchmark_names
    }

    # 5. MAPE
    mape_scores = {"Chronos": mape(actuals.values, chronos_median[:len(test_dates)])}
    for name, preds in benchmark_series.items():
        mape_scores[name] = mape(actuals.values, preds.values)

    # 6. Result table
    result_df = pd.DataFrame({"Actual": actuals.values}, index=test_dates)
    result_df["Chronos"] = chronos_median[:len(test_dates)]
    for name, preds in benchmark_series.items():
        result_df[name] = preds.values

    return {
        "label":      horizon.get("label", horizon["test_start"]),
        "result_df":  result_df,
        "mape":       mape_scores,
        "quantiles":  forecast,
    }

## 11. Full pipeline

Loops `run_one_horizon` over all three horizons and assembles a MAPE summary
DataFrame — rows = horizons, columns = Chronos + benchmark names.

In [ ]:
def run_pipeline(
    *,
    data_df: pd.DataFrame,
    metric_df: pd.DataFrame,
    horizons: list[dict],
    runtime,
    endpoint_name: str,
    target_col: str,
    portfolio: str,
    sub_portfolio: str,
    covariate_cols: Optional[list[str]] = None,
    ma_window: int = MA_WINDOW,
    prediction_length: int = PREDICTION_LENGTH,
    quantiles: tuple[float, ...] = DEFAULT_QUANTILES,
) -> tuple[dict, pd.DataFrame]:
    """
    Returns
    -------
    per_horizon  : {label: result dict from run_one_horizon}
    mape_summary : DataFrame  rows=horizons  cols=Chronos + benchmarks
    """
    per_horizon = {}
    for h in horizons:
        label = h.get("label", h["test_start"])
        print(f"--- Running horizon: {label} ---")
        per_horizon[label] = run_one_horizon(
            data_df=data_df,
            metric_df=metric_df,
            horizon=h,
            runtime=runtime,
            endpoint_name=endpoint_name,
            target_col=target_col,
            portfolio=portfolio,
            sub_portfolio=sub_portfolio,
            covariate_cols=covariate_cols,
            ma_window=ma_window,
            prediction_length=prediction_length,
            quantiles=quantiles,
        )
        print(f"    MAPE: {per_horizon[label]['mape']}\n")

    mape_summary = pd.DataFrame(
        {label: res["mape"] for label, res in per_horizon.items()}
    ).T
    mape_summary.index.name = "horizon"
    return per_horizon, mape_summary

---

## 12. Select covariates

Plug your feature-selection function in here. The placeholder list is illustrative.

In [ ]:
# covariate_cols = pick_best_features(prepared_train_df, target_col=FORECASTING_METRIC)
covariate_cols = ["target_lag_1", "sma_3", "ema_5", "month_sin", "month_cos"]
print("Covariates:", covariate_cols)

## 13. Run the pipeline

In [ ]:
runtime = boto3.client("sagemaker-runtime")

per_horizon, mape_summary = run_pipeline(
    data_df=data_df,
    metric_df=metric_df,
    horizons=HORIZONS,
    runtime=runtime,
    endpoint_name=ENDPOINT_NAME,
    target_col=FORECASTING_METRIC,
    portfolio=PORTFOLIO,
    sub_portfolio=SUB_PORTFOLIO,
    covariate_cols=covariate_cols,
)

## 14. MAPE summary

Rows = horizons. Columns = Chronos + the benchmark names defined per horizon.
Lower is better.

In [ ]:
mape_summary.round(2)

## 15. Per-horizon forecast tables

Actuals vs Chronos vs each benchmark, indexed by test month.

In [ ]:
for label, res in per_horizon.items():
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    display(res["result_df"].round(2))